In [1]:
# In RNA-LLM paper PDB-RNA dataset is the same as 
# TR1-TV1-TS1 dataset of SPOT-RNA2
# with 120, 30 and 62 sequences respectively

# This notebook just convert structures to bpseq and sequence to fasta
# to be used in RedFold

import os
import ast
import pandas as pd

In [2]:
DATA_PATH = "/DATA/lncRNA/data/rna-llm/"

# RNA format conversion

In [3]:
# Convert a CT file with RNA sequence and structure to Bpseq format
def ct_to_bpseq(ct_file_path, bpseq_file_path):
    """
    Converts a CT file with RNA sequence and structure to Bpseq format.
    The Bpseq format is written to a file specified by `bpseq_file_path`.
    
    Args:
    - ct_file_path (str): Path to the input CT file.
    - bpseq_file_path (str): Path to the output Bpseq file.
    """
    with open(ct_file_path, "r") as ct_file, open(bpseq_file_path, "w") as bpseq_file:
        # Read the first line of the CT file to get the sequence length
        line = ct_file.readline()
        seq_length = int(line.strip().split()[0])
        
        # Initialize lists for the nucleotide names, positions, and base pairs
        nucleotide_names = [None] * (seq_length + 1) # use None for the 0th position
        nucleotide_positions = [None] * (seq_length + 1)
        base_pairs = [None] * (seq_length + 1)
        
        # Read the nucleotide information from the CT file
        for i in range(seq_length):
            line = ct_file.readline()
            fields = line.strip().split()
            nucleotide_position = int(fields[0])
            nucleotide_name = fields[1]
            base_pair = int(fields[4])
            nucleotide_names[nucleotide_position] = nucleotide_name
            nucleotide_positions[nucleotide_position] = nucleotide_position
            base_pairs[nucleotide_position] = base_pair
            
        # Write the nucleotide information in Bpseq format to the output file
        for i in range(1, seq_length + 1):
#ddd            bpseq_file.write(f"{i} {nucleotide_names[i]} {nucleotide_positions[i]} {base_pairs[i]}\n")
            bpseq_file.write(f"{i} {nucleotide_names[i]} {base_pairs[i]}\n")

In [6]:
#ct_to_bpseq("bkp/5s_Achromobacter-xylosoxidans-1.ct", "5s_Achromobacter-xylosoxidans-1.bseq")

In [4]:
def write_ct(fname, seqid, seq, base_pairs):
    """Write ct file from sequence and base pairs. Base_pairs should be 1-based and unique per nt"""
    base_pairs_dict = {}
    for bp in base_pairs:
        base_pairs_dict[bp[0]] = bp[1]
        base_pairs_dict[bp[1]] = bp[0]

    with open(fname, "w") as fout:
        fout.write(f"{len(seq)} {seqid}\n")
        for k, n in enumerate(seq):
            fout.write(f"{k+1} {n} {k} {k+2} {base_pairs_dict.get(k+1, 0)} {k+1}\n")

def remove_non_canonical(base_pairs, seq):
    """Remove non-canonical base pairs from a list of base pairs"""
    canonical = ['AU', 'UA', 'CG', 'GC', 'GU', 'UG']
    
    bp_canonical = []
    for bp in base_pairs:
        if seq[bp[0]-1] + seq[bp[1]-1] in canonical:
            bp_canonical.append(bp)
    
    return bp_canonical


# Conversion

In [5]:
dref = pd.read_csv(DATA_PATH + "PDB-RNA.csv")
dref.set_index("id", inplace=True)
display(dref.head())

,sequence,base_pairs,len
id,,,
1ddy-1-A,GGAACCGGUGCGCAUAACCACCUCAGUGCGAGCAA,"[[7, 22], [8, 21], [8, 25], [9, 20], [10, 18],...",35
1dk1-1-B,GGGCGGCCUUCGGGCUAGACGGUGGGAGAGGCUUCGGCUGGUCCAC...,"[[1, 57], [2, 56], [3, 55], [4, 54], [6, 15], ...",57
1dul-1-B,GGCUCUGUUUACCAGGUCAGGUCCGAAAGGAAGCAGCCAAGGCAGAGCC,"[[1, 49], [2, 48], [3, 47], [4, 46], [5, 45], ...",49
1ffk-1-9,UUAGGCGGCCACAGCGGUGGGGUUGCCUCCCGUACCCAUCCCGAAC...,"[[3, 21], [3, 25], [4, 119], [5, 118], [6, 117...",122
1ffy-1-T,GGGCUUGUAGCUCAGGUGGUUAGAGCGCACCCCUGAUAAGGGUGAG...,"[[1, 73], [2, 72], [3, 71], [4, 70], [5, 69], ...",75


In [6]:
splits = pd.read_csv(DATA_PATH + "PDB-RNA_splits.csv")
display(splits.head())


,id,partition
0,1ddy-1-A,train
1,1dk1-1-B,train
2,1dul-1-B,train
3,1ffk-1-9,train
4,1ffy-1-T,train


In [8]:
OUT_PATH = "/DATA/lncRNA/redfold/PDB-RNA"
if not os.path.exists(OUT_PATH): os.makedirs(OUT_PATH)

parts = splits.partition.unique()
    
for part in parts:
    dirp = OUT_PATH + "/" + part + "/"
    if not os.path.exists(dirp): os.makedirs(dirp)

    seq_ids = list(splits.loc[(splits.partition==part)].id)

    for seq_id in seq_ids:
        if dref.loc[seq_id].len <= 512:
            seqA = dref.loc[seq_id]["sequence"].replace("N","A") # igual da ERROR porque quedan pares incorrectos como AG
            seqA = seqA.replace("T","U") # PDB-RNA has a T instead of U in sequence 3d2v-1-A! 😤
            if dref.loc[seq_id]["sequence"].count("N")==0: # <<<<<<<<<<<<< ELIMINO
                if part == "test": # convert to fasta
                    with open(dirp + seq_id + ".fasta", "w") as tmpfile:
                        tmpfile.write(">" + seq_id + "\n" + seqA + "\n")
                else:              # convert to bpseq
                    # redfold only accepts canonical base pairs!!
                    bp_canonical = remove_non_canonical(ast.literal_eval(dref.loc[seq_id]["base_pairs"]), seqA)
                    write_ct("tmp.ct", seq_id, seqA, bp_canonical)
                    # the use of intermediate CT file was inherited from the original code using RNAstructure conversor
                    ct_to_bpseq("tmp.ct", dirp + seq_id + ".bpseq")
        else:
            print("WARNING L>512:", seq_id)